## 🎯 Learning Outcomes
### By the end of this module, you will be able to

- Understand different file modes (`'r'`, `'w'`, `'a'`, `'rb'`, `'wb'`) for opening files.
- Read file content using `read()`, `readline()`, `readlines()`, and iteration.
- Write file content using `write()` and `writelines()`.
- Utilize the `with` statement for safe and automatic file closing (context manager protocol).
- Work with binary files using `'rb'` and `'wb'` modes.
- Comprehend Python's exception hierarchy.
- Implement `try`, `except`, `else`, and `finally` blocks for robust error handling.
- Catch specific and broad exceptions appropriately.
- Raise and re-raise exceptions, including using `raise ... from` for exception chaining.
- Define and use custom exception classes for application-specific error handling.

## Part 1 — File I/O

### 1.1 Opening Files with `open()` — File Modes

The built-in `open(filename, mode)` function returns a **file object**. The `mode` string controls how the file is opened:

| Mode | Meaning |
|------|---------|
| `'r'` | Read (text) — default; file must exist |
| `'w'` | Write (text) — creates or **truncates** |
| `'a'` | Append (text) — creates or appends |
| `'rb'` | Read binary |
| `'wb'` | Write binary |
| `'r+'` | Read + write (text) |
| `'x'` | Exclusive creation — fails if file exists |

In [1]:
# Create a sample text file to work with throughout this notebook
sample_text = """Python is a versatile language.
File I/O is a core skill.
Error handling keeps programs robust.
Always close your files!
The with statement handles this for us.
"""

with open("sample.txt", "w") as f:
    f.write(sample_text)

print("sample.txt created successfully.") #In Google Colab, you will find the file located in `content` folder.

sample.txt created successfully.


### 1.2 Reading Files

Three main methods + iteration:

In [2]:
# read() — reads the entire file as a single string
f = open("sample.txt", "r")
content = f.read()
f.close()  # must close manually when not using 'with'

print("=== read() ===")
print(repr(content))

=== read() ===
'Python is a versatile language.\nFile I/O is a core skill.\nError handling keeps programs robust.\nAlways close your files!\nThe with statement handles this for us.\n'


In [3]:
# readline() — reads one line at a time (including the newline character)
f = open("sample.txt", "r")
print("=== readline() ===")
print(repr(f.readline()))  # line 1
print(repr(f.readline()))  # line 2
f.close()

=== readline() ===
'Python is a versatile language.\n'
'File I/O is a core skill.\n'


In [4]:
# readlines() — reads all lines into a list
f = open("sample.txt", "r")
lines = f.readlines()
f.close()

print("=== readlines() ===")
for i, line in enumerate(lines, 1):
    print(f"  Line {i}: {repr(line)}")

=== readlines() ===
  Line 1: 'Python is a versatile language.\n'
  Line 2: 'File I/O is a core skill.\n'
  Line 3: 'Error handling keeps programs robust.\n'
  Line 4: 'Always close your files!\n'
  Line 5: 'The with statement handles this for us.\n'


In [5]:
# Iteration — most Pythonic: the file object itself is an iterator
# Memory-efficient for large files (reads one line at a time)
print("=== Iterating over file ===")
with open("sample.txt", "r") as f:
    for line in f:
        print(line, end="")  # line already has \n

=== Iterating over file ===
Python is a versatile language.
File I/O is a core skill.
Error handling keeps programs robust.
Always close your files!
The with statement handles this for us.


### 1.3 Writing Files

In [6]:
# write() — writes a single string; does NOT add a newline automatically
with open("output.txt", "w") as f:
    f.write("First line\n")
    f.write("Second line\n")
    chars_written = f.write("Third line\n")  # returns number of chars written

print(f"Last write() returned: {chars_written} characters")

# Verify
with open("output.txt") as f:
    print(f.read())

Last write() returned: 11 characters
First line
Second line
Third line



In [7]:
# writelines() — writes an iterable of strings (no automatic newlines added!)
lines = ["Alpha\n", "Beta\n", "Gamma\n"]

with open("output2.txt", "w") as f:
    f.writelines(lines)

with open("output2.txt") as f:
    print(f.read())

Alpha
Beta
Gamma



In [8]:
# Append mode — adds to existing file without truncating
with open("output.txt", "a") as f:
    f.write("Fourth line (appended)\n")

with open("output.txt") as f:
    print(f.read())

First line
Second line
Third line
Fourth line (appended)



### 1.4 The `with` Statement — Context Manager Protocol

The `with` statement ensures that `file.close()` is **always** called — even if an exception occurs inside the block. This is the **recommended** way to work with files.

```
with open(path, mode) as f:
    # work with f
# f is automatically closed here
```

Under the hood, `open()` returns an object implementing the **context manager protocol** (`__enter__` / `__exit__`). The `__exit__` method calls `close()` unconditionally.

In [9]:
# Without 'with' — risky: if an exception occurs before close(), file leaks
f = open("sample.txt")
try:
    data = f.read()
finally:
    f.close()  # manual equivalent of what 'with' does for you

# With 'with' — clean, safe, idiomatic
with open("sample.txt") as f:
    data = f.read()
# f is closed here — even if read() raised an exception

print(f"File closed after 'with' block: {f.closed}")

File closed after 'with' block: True


In [10]:
# Opening multiple files at once with a single 'with'
with open("sample.txt") as src, open("copy.txt", "w") as dst:
    dst.write(src.read())

print("File copied. Both handles closed:", src.closed, dst.closed)

File copied. Both handles closed: True True


### 1.5 Binary Files — `rb` and `wb`

Use binary mode when working with images, audio, executables, or any non-text data. In binary mode:
- Data is read/written as `bytes`, not `str`
- No newline translation occurs

In [11]:
import struct

# Write some raw binary data
with open("data.bin", "wb") as f:
    f.write(b"\x89PNG")          # PNG magic bytes
    f.write(struct.pack(">I", 42))  # big-endian unsigned int

# Read it back
with open("data.bin", "rb") as f:
    raw = f.read()

print(f"Raw bytes : {raw}")
print(f"Hex       : {raw.hex()}")
print(f"First 4   : {raw[:4]}")

Raw bytes : b'\x89PNG\x00\x00\x00*'
Hex       : 89504e470000002a
First 4   : b'\x89PNG'


In [12]:
# Practical example: copy a binary file in chunks (memory-efficient)
CHUNK_SIZE = 4096  # 4 KB at a time

def binary_copy(src_path: str, dst_path: str) -> int:
    """Copy a binary file in chunks. Returns total bytes copied."""
    total = 0
    with open(src_path, "rb") as src, open(dst_path, "wb") as dst:
        while chunk := src.read(CHUNK_SIZE):   # walrus operator
            dst.write(chunk)
            total += len(chunk)
    return total

copied = binary_copy("data.bin", "data_copy.bin")
print(f"Copied {copied} bytes")

Copied 8 bytes


---

## Part 2 — Error Handling & Exceptions

### 2.1 The Exception Hierarchy

All Python exceptions descend from `BaseException`. The subtree you work with most:

```
BaseException
 ├── SystemExit
 ├── KeyboardInterrupt
 ├── GeneratorExit
 └── Exception             ← catch this in almost all cases
      ├── ArithmeticError
      │    ├── ZeroDivisionError
      │    └── OverflowError
      ├── LookupError
      │    ├── IndexError
      │    └── KeyError
      ├── OSError / IOError
      │    ├── FileNotFoundError
      │    ├── PermissionError
      │    └── IsADirectoryError
      ├── ValueError
      ├── TypeError
      ├── AttributeError
      ├── NameError
      ├── RuntimeError
      └── StopIteration
```

In [13]:
# Inspect the MRO (method resolution order) of a specific exception
print("FileNotFoundError MRO:")
for cls in FileNotFoundError.__mro__:
    print(f"  {cls.__name__}")

FileNotFoundError MRO:
  FileNotFoundError
  OSError
  Exception
  BaseException
  object


### 2.2 `try`, `except`, `else`, `finally` — Semantics

```python
try:
    # code that might raise
except SomeError as e:
    # runs ONLY if SomeError (or subclass) was raised
else:
    # runs ONLY if NO exception was raised in try
finally:
    # ALWAYS runs — exception or not
```

In [14]:
def demonstrate_blocks(value):
    print(f"\n--- Input: {value!r} ---")
    try:
        result = 10 / int(value)
    except ZeroDivisionError:
        print("EXCEPT  : division by zero!")
    except ValueError:
        print("EXCEPT  : can't convert to int!")
    else:
        print(f"ELSE    : success, result = {result:.2f}")
    finally:
        print("FINALLY : this always runs")

demonstrate_blocks(2)       # success path → else + finally
demonstrate_blocks(0)       # ZeroDivisionError → except + finally
demonstrate_blocks("abc")   # ValueError → except + finally


--- Input: 2 ---
ELSE    : success, result = 5.00
FINALLY : this always runs

--- Input: 0 ---
EXCEPT  : division by zero!
FINALLY : this always runs

--- Input: 'abc' ---
EXCEPT  : can't convert to int!
FINALLY : this always runs


### 2.3 Catching Specific vs. Broad Exceptions

In [15]:
# ✅ Catching specific exceptions — preferred
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return float('inf')

# ✅ Multiple except clauses
def safe_parse(text, key):
    try:
        import json
        data = json.loads(text)
        return data[key]
    except json.JSONDecodeError as e:
        print(f"Bad JSON at line {e.lineno}: {e.msg}")
    except KeyError:
        print(f"Key {key!r} not found in JSON object")
    return None

# ✅ Catching a parent class catches all its children
def safe_read(path):
    try:
        with open(path) as f:
            return f.read()
    except OSError as e:          # catches FileNotFoundError, PermissionError, etc.
        print(f"OS error [{e.errno}]: {e.strerror} — {path}")
    return None

print(safe_divide(5, 0))
safe_parse('{"x": 1}', "y")
safe_read("no_such_file.txt")

inf
Key 'y' not found in JSON object
OS error [2]: No such file or directory — no_such_file.txt


In [16]:
# ❌ Anti-pattern: bare except (catches EVERYTHING including KeyboardInterrupt, SystemExit)
def bad_example():
    try:
        risky_operation = 1 / 0
    except:                 # ← hides bugs, swallows Ctrl+C
        pass                # silent failure — worst of all worlds

# ❌ Also avoid catching Exception too broadly without re-raising
def also_bad(data):
    try:
        process(data)
    except Exception:       # too wide if you just 'pass'
        pass

# ✅ If you must catch broadly, at least log it
import logging
def acceptable_broad_catch(data):
    try:
        return int(data) * 2
    except Exception as e:
        logging.exception("Unexpected error processing data")  # logs full traceback
        raise  # re-raise so the caller knows

print("Bare except demo complete (no exceptions escaped)")

Bare except demo complete (no exceptions escaped)


### 2.4 Raising Exceptions: `raise`, re-raising, and `raise ... from`

In [17]:
# raise — create and throw an exception
def set_age(age):
    if not isinstance(age, int):
        raise TypeError(f"age must be int, got {type(age).__name__}")
    if age < 0 or age > 150:
        raise ValueError(f"age {age} is out of reasonable range [0, 150]")
    return age

try:
    set_age("forty")
except TypeError as e:
    print(f"TypeError: {e}")

try:
    set_age(-5)
except ValueError as e:
    print(f"ValueError: {e}")

TypeError: age must be int, got str
ValueError: age -5 is out of reasonable range [0, 150]


In [18]:
# Bare raise — re-raises the current exception, preserving the original traceback
def process_file(path):
    try:
        with open(path) as f:
            return f.read()
    except FileNotFoundError:
        print(f"[LOG] File not found: {path}")
        raise  # re-raise — caller gets the original exception

try:
    process_file("missing.txt")
except FileNotFoundError:
    print("Caller caught the re-raised FileNotFoundError")

[LOG] File not found: missing.txt
Caller caught the re-raised FileNotFoundError


In [19]:
# raise ... from — exception chaining: wraps a low-level error in a higher-level one
# The original exception is stored as __cause__ and shown in the traceback

class ConfigError(Exception):
    pass

def load_config(path):
    try:
        with open(path) as f:
            import json
            return json.load(f)
    except FileNotFoundError as e:
        raise ConfigError(f"Config file not found: {path}") from e
    except json.JSONDecodeError as e:
        raise ConfigError(f"Invalid JSON in config: {path}") from e

try:
    load_config("config.json")
except ConfigError as e:
    print(f"ConfigError : {e}")
    print(f"Caused by   : {e.__cause__}")

ConfigError : Config file not found: config.json
Caused by   : [Errno 2] No such file or directory: 'config.json'


In [20]:
# raise ... from None — suppress chaining (hides the original exception)
def strict_lookup(d, key):
    try:
        return d[key]
    except KeyError:
        raise ValueError(f"Unknown key: {key!r}") from None  # __cause__ suppressed

try:
    strict_lookup({"a": 1}, "z")
except ValueError as e:
    print(f"ValueError : {e}")
    print(f"__cause__  : {e.__cause__}")  # None

ValueError : Unknown key: 'z'
__cause__  : None


### 2.5 Custom Exception Classes

Inherit from `Exception` (or a more specific subclass) to create your own exception types. This enables callers to catch *your* exceptions selectively.

In [21]:
# ── Minimal custom exception ──────────────────────────────────────────────────
class AppError(Exception):
    """Base exception for this application."""

class DatabaseError(AppError):
    """Raised when a database operation fails."""

class NetworkError(AppError):
    """Raised when a network operation fails."""

# ── Custom exception with extra attributes ────────────────────────────────────
class ValidationError(AppError):
    """Raised when input validation fails."""

    def __init__(self, field: str, value, message: str):
        self.field = field
        self.value = value
        self.message = message
        # Pass a human-readable string to the base Exception
        super().__init__(f"Validation failed on '{field}': {message} (got {value!r})")

# ── Usage ─────────────────────────────────────────────────────────────────────
def validate_email(email):
    if "@" not in email:
        raise ValidationError("email", email, "must contain '@'")
    return email

try:
    validate_email("not-an-email")
except ValidationError as e:
    print(f"Caught: {e}")
    print(f"  field : {e.field}")
    print(f"  value : {e.value!r}")
    print(f"  msg   : {e.message}")

Caught: Validation failed on 'email': must contain '@' (got 'not-an-email')
  field : email
  value : 'not-an-email'
  msg   : must contain '@'


In [22]:
# Catching a parent class catches all subclasses
def run_operation(op_type):
    if op_type == "db":
        raise DatabaseError("Connection refused")
    elif op_type == "net":
        raise NetworkError("Timeout after 30s")

for op in ["db", "net"]:
    try:
        run_operation(op)
    except DatabaseError as e:
        print(f"DB issue: {e}")
    except NetworkError as e:
        print(f"Net issue: {e}")

# Catch all app errors at once
for op in ["db", "net"]:
    try:
        run_operation(op)
    except AppError as e:
        print(f"App error ({type(e).__name__}): {e}")

DB issue: Connection refused
Net issue: Timeout after 30s
App error (DatabaseError): Connection refused
App error (NetworkError): Timeout after 30s


---

## Part 3 — Putting It All Together: Robust File Processing

A realistic example that combines file I/O with proper error handling.

In [23]:
import json
import os
from pathlib import Path

# ── Custom exceptions for our pipeline ────────────────────────────────────────
class PipelineError(Exception):
    """Base error for the data pipeline."""

class InputError(PipelineError):
    """Problem with input file."""

class ProcessingError(PipelineError):
    """Problem while processing data."""

# ── Utility functions ─────────────────────────────────────────────────────────
def read_json_file(path: str) -> dict:
    """Read a JSON file and return parsed data."""
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError as e:
        raise InputError(f"File not found: {path}") from e
    except PermissionError as e:
        raise InputError(f"No read permission: {path}") from e
    except json.JSONDecodeError as e:
        raise InputError(f"Invalid JSON in {path} at line {e.lineno}") from e

def write_report(path: str, data: dict) -> None:
    """Write a summary report to a text file."""
    try:
        with open(path, "w", encoding="utf-8") as f:
            f.write("=== REPORT ===\n")
            for k, v in data.items():
                f.write(f"{k}: {v}\n")
    except OSError as e:
        raise PipelineError(f"Could not write report to {path}") from e

def process_records(records: list) -> dict:
    """Process a list of numeric records."""
    if not records:
        raise ProcessingError("Record list is empty")
    try:
        values = [float(r) for r in records]
    except (TypeError, ValueError) as e:
        raise ProcessingError(f"Non-numeric record encountered") from e
    return {
        "count": len(values),
        "total": sum(values),
        "mean": sum(values) / len(values),
        "min": min(values),
        "max": max(values),
    }

# ── Main pipeline ─────────────────────────────────────────────────────────────
def run_pipeline(input_path: str, output_path: str) -> None:
    print(f"Starting pipeline: {input_path} → {output_path}")
    try:
        raw = read_json_file(input_path)
        records = raw.get("values", [])
        summary = process_records(records)
        write_report(output_path, summary)
    except InputError as e:
        print(f"[INPUT ERROR] {e}")
    except ProcessingError as e:
        print(f"[PROCESSING ERROR] {e}")
    except PipelineError as e:
        print(f"[PIPELINE ERROR] {e}")
    else:
        print(f"Pipeline complete. Report written to: {output_path}")
    finally:
        print("Pipeline finished (cleanup would go here)")

# ── Create test data and run ───────────────────────────────────────────────────
test_data = {"values": [12, 7, 3, 44, 8, 21, 5, 17]}
with open("input.json", "w") as f:
    json.dump(test_data, f)

run_pipeline("input.json", "report.txt")

print()
with open("report.txt") as f:
    print(f.read())

# Error case
run_pipeline("nonexistent.json", "report.txt")

Starting pipeline: input.json → report.txt
Pipeline complete. Report written to: report.txt
Pipeline finished (cleanup would go here)

=== REPORT ===
count: 8
total: 117.0
mean: 14.625
min: 3.0
max: 44.0

Starting pipeline: nonexistent.json → report.txt
[INPUT ERROR] File not found: nonexistent.json
Pipeline finished (cleanup would go here)


---

# Summary

| Concept | Key Points |
|---|---|
| **File I/O Cheatsheet** ||
| Read entire file | `open(p).read()` |
| Read lines as list | `open(p).readlines()` |
| Iterate lines | `for line in open(p)` |
| Write string | `open(p, 'w').write(s)` |
| Append | `open(p, 'a').write(s)` |
| Safe open | `with open(p) as f:` |
| Binary read | `open(p, 'rb').read()` |
| **Exception Handling Cheatsheet** ||
| `except ValueError` | Expected specific error |
| `except (A, B) as e` | Multiple types, same handler |
| `except OSError as e` | Any OS/IO error |
| `else` block | Code that runs only on success |
| `finally` block | Cleanup that always runs |
| `raise` | Re-raise current exception |
| `raise X from Y` | Wrap low-level error in high-level |
| `raise X from None` | Suppress chained context |
| Custom exception | `class MyError(Exception): ...` |

### Contributed by: Abdulhadi Zubailah